# Notebook 01 — NumPy Foundations for LLM Engineering

    ## Learning objectives

    - Master shapes, axes, broadcasting, indexing, and dtypes
- Implement stable probability and attention operations
- Debug vectorized numerical code with explicit invariants

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 1.1 Arrays are typed strided views

Shape names the logical axes while strides map indices to memory. LLM code repeatedly moves among batch, sequence, head, and feature axes, so most silent failures are axis errors. Reshape may share storage; transpose usually changes strides; advanced indexing generally copies. Inspect shape, dtype, strides, byte count, and memory sharing. Token IDs remain integers while activations are floating point. Broadcasting aligns from the right and can create an accidental quadratic temporary when singleton dimensions are missing. Write shape comments and assertions at every interface rather than relying on a plausible output.


In [ ]:
import numpy as np
x=np.arange(24,dtype=np.float32).reshape(2,3,4); y=x.transpose(0,2,1)
print(x.shape,x.strides,y.shape,y.strides,np.shares_memory(x,y))


## 1.2 Vectorization and tensor algebra

Embedding lookup is row indexing in a vocabulary-by-width matrix. Linear projections contract a feature axis. `einsum` makes named contractions explicit and is useful for deriving batched multi-head operations, even when production kernels use specialized matrix multiplication. Reductions must specify the axis and often retain dimensions for later broadcasting. Verify a vectorized implementation against a tiny loop, then profile warmed operations. Concise expressions can still allocate large intermediates or traverse memory repeatedly; vectorization is a performance tool only when memory behavior is understood.


In [ ]:
rng=np.random.default_rng(42); e=rng.normal(size=(17,8)); ids=np.array([[1,4,1],[3,2,9]])
h=e[ids]; w=rng.normal(size=(8,12)); print(h.shape,np.einsum("btf,fd->btd",h,w).shape)


## 1.3 Stable probability computation

Softmax must subtract the row maximum before exponentiation. Log-softmax uses log-sum-exp rather than taking a logarithm after probabilities have underflowed. Cross-entropy selects negative target log-probabilities and averages only eligible labels. Perplexity exponentiates mean token loss and is comparable only with the same tokenizer, corpus, boundaries, and masks. Floating tests use tolerances and invariants: finite nonnegative probabilities sum to one, a constant logit shift changes nothing, and ignored positions contribute neither numerator nor denominator.


In [ ]:
def log_softmax(x):
 m=x.max(-1,keepdims=True); z=x-m; return z-np.log(np.exp(z).sum(-1,keepdims=True))
l=np.array([[1000.,1001.,999.]]); print(np.exp(log_softmax(l)), -log_softmax(l)[0,1])


## 1.4 Attention and scientific practice

Scaled dot-product attention contracts queries with keys, divides by square root of head width, applies a causal or padding mask, softmaxes over key positions, and mixes values. This transparent NumPy form materializes the quadratic score matrix and is a reference rather than a long-context implementation. Use explicit random generators, separate data streams, and avoid reseeding loops. Test extreme logits, batch or sequence length one, empty selections, masked rows, and dtype changes. Record shapes and timings so later PyTorch and fused implementations can be checked against the same mathematical result.


In [ ]:
def softmax(x):
 z=x-x.max(-1,keepdims=True); e=np.exp(z); return e/e.sum(-1,keepdims=True)
rng=np.random.default_rng(7); q=rng.normal(size=(1,2,4,8)); k=rng.normal(size=q.shape); v=rng.normal(size=q.shape)
s=np.einsum("bhtd,bhsd->bhts",q,k)/np.sqrt(8); s=np.where(np.tril(np.ones((4,4),bool))[None,None],s,-np.inf)
p=softmax(s); print(np.einsum("bhts,bhsd->bhtd",p,v).shape,p.sum(-1))


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## Exercises

    1. Implement masked cross-entropy.
2. Prove a vectorized attention result matches loops.
3. Diagnose three broadcasting bugs.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
